In [ ]:
import requests
import pandas as pd
import numpy as np

# API URL für historische Wetterdaten in New York (Jahr 2023 als Beispiel)
# Wir holen: Wetter-Code (Sonne, Regen etc.), Durchschnittstemperatur und Niederschlagssumme
url = "https://archive-api.open-meteo.com/v1/archive?latitude=40.7128&longitude=-74.0060&start_date=2023-01-01&end_date=2023-12-31&daily=weather_code,temperature_2m_mean,precipitation_sum&timezone=America%2FNew_York"

# Wir schauen ob die API reagiert:

In [ ]:
response = requests.get(url)

if response.status_code == 200:
    print("Erfolg! Die API antwortet mit Status 200.")
else:
    print(f"Fehler! Status Code: {response.status_code}")

# Wir fragen ab was uns die API liefert:

In [ ]:
# Die API liefert ein JSON. Wir extrahieren den Teil "daily", in dem die Tagesdaten stecken
data = response.json()
daily_data = data['daily']

# Wir wandeln das JSON-Dictionary in einen sauberen Pandas DataFrame um (Extract-Phase)
df_weather = pd.DataFrame(daily_data)

print("Die ersten 5 Zeilen der rohen API-Daten:")
display(df_weather.head())

# Wir schauen uns die Datentypen an:

In [ ]:
print("Datentypen der einzelnen Spalten:")
print(df_weather.dtypes)

print("\nZusätzliche Infos (Anzahl der Zeilen und Non-Null Werte):")
df_weather.info()

# Wir überprüfen die Daten auf mögliche Verunreinigungen:

In [ ]:
print("Anzahl der fehlenden Werte (NaN) pro Spalte:")
print(df_weather.isnull().sum())

print(f"\nAnzahl der komplett identischen Duplikate: {df_weather.duplicated().sum()}")

# Kurzer Check, ob die Werte realistisch sind (z.B. Temperatur nicht bei +500 Grad)
print("\nStatistische Übersicht (Min/Max Werte prüfen):")
display(df_weather.describe())

# Wir bereinigen die Daten:

In [ ]:
# 1. Wir benennen die Spalten um, damit sie lesbar und bereit für das Star Schema sind
df_weather.rename(columns={
    'time': 'Date',
    'weather_code': 'Weather_Code',
    'temperature_2m_mean': 'Avg_Temperature',
    'precipitation_sum': 'Precipitation'
}, inplace=True)

# 2. Den Zeit-String in ein echtes Datum-Objekt (datetime) umwandeln
df_weather['Date'] = pd.to_datetime(df_weather['Date'])

# 3. Fehlende Werte behandeln (falls es z.B. Lücken in den Sensordaten gab)
# Wir füllen leere Temperatur- oder Niederschlagswerte einfach mit 0 auf
df_weather.fillna(0, inplace=True)

print("Daten sind bereinigt!")
display(df_weather.head())

In [ ]:
# HIERHIN: Wir speichern sie als "Staging-Tabelle" (Zwischenspeicher) lokal ab.
# Das ist Best-Practice im Data Warehousing, bevor man sie endgültig verknüpft.
staging_pfad = "nyc_weather_staging_2023.csv"
df_weather.to_csv(staging_pfad, index=False)

print(f"Daten erfolgreich in den Zwischenspeicher '{staging_pfad}' geladen.")

In [ ]:
# DORTHIN: Wir laden sie in unsere finale Zieldimension (Dim_Date) für das DWH[cite: 327, 331].
# Zuerst erstellen wir testweise einen Basis-Kalender für Dim_Date
df_dim_date = pd.DataFrame({'Date': pd.date_range(start='2023-01-01', end='2023-12-31')})

# Jetzt verschmelzen (mergen) wir die Wetterdaten mit unserer Dim_Date
# Das ist der Moment, wo das Wetter in euer Star Schema integriert wird!
df_dim_date_final = pd.merge(df_dim_date, df_weather, on='Date', how='left')

print("Wetterdaten erfolgreich in die finale Dim_Date Tabelle geladen!")
display(df_dim_date_final.head())

In [ ]:
# NOTWENDIG FÜR POWER BI: Erstellung eines numerischen Primary Keys (Date_ID).
# Aus '2023-01-15' wird die Zahl 20230115. Das macht die Datenbank und Power BI viel schneller.

df_dim_date_final['Date_ID'] = df_dim_date_final['Date'].dt.strftime('%Y%m%d').astype(int)

# Wir sortieren die Spalten etwas schöner
df_dim_date_final = df_dim_date_final[['Date_ID', 'Date', 'Weather_Code', 'Avg_Temperature', 'Precipitation']]

display(df_dim_date_final.head())

In [ ]:
# NOTWENDIG FÜR DAS MARKETING-DASHBOARD: Übersetzung der numerischen Wetter-Codes.
# Die API liefert uns "Weather Codes" (WMO). 0 ist Sonne, 71 ist Schnee. 
# Für ein beschreibendes Dimension-Attribut brauchen wir aber verständlichen Text[cite: 662]!

# Wir definieren ein Mapping-Dictionary (Auszug der wichtigsten WMO Codes)
wmo_mapping = {
    0: 'Clear sky',
    1: 'Mainly clear',
    2: 'Partly cloudy',
    3: 'Overcast',
    51: 'Light Drizzle',
    53: 'Moderate Drizzle',
    61: 'Slight Rain',
    63: 'Moderate Rain',
    65: 'Heavy Rain',
    71: 'Slight Snow',
    73: 'Moderate Snow',
    75: 'Heavy Snow'
}

# Wir mappen die Texte auf die Codes. Falls ein Code fehlt, nennen wir es 'Unknown'
df_dim_date_final['Weather_Condition'] = df_dim_date_final['Weather_Code'].map(wmo_mapping).fillna('Unknown')

# Alte Code-Spalte löschen, wir brauchen nur den sauberen Text für Power BI
df_dim_date_final.drop('Weather_Code', axis=1, inplace=True)

print("Finale, Power-BI-bereite Dim_Date Tabelle mit integriertem Wetter:")
display(df_dim_date_final.head())